# LLM Fine-Tuning & Retrieval-Augmented Generation
Cells marked **[My implementation]** are my code; cells marked **[Course-provided]** come from the course's starter notebook. See the README for details.

## Setup

In [ ]:
# [Course-provided]
# Verify that CUDA / T4 GPU is available before running model cells.
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

### Step 1: Install all dependencies

In [ ]:
# [Course-provided]
# Install all project dependencies (bitsandbytes, transformers, trl, haystack, chromadb, etc.).
# ⚠️  Re-run this cell after every runtime restart — pip installs are lost when the session ends.
# %reset -f (used at the start of Section 3) only clears Python variables;
# packages remain installed and do NOT need to be reinstalled after %reset -f.

!pip install bitsandbytes>=0.39.0
!pip install --upgrade accelerate transformers datasets peft trl
!pip install streamlit nltk
!npm install -g localtunnel
!pip install sentence-transformers
!pip install chromadb
!pip install haystack-ai elasticsearch-haystack
!npm install -g localtunnel

### Step 2: Download lecture transcripts

In [ ]:
# [Course-provided]
# Unzip the 23 CS 639 lecture transcript .txt files from the course GitHub.
!unzip transcripts.zip -d transcripts/

### Step 3: HuggingFace login

In [ ]:
# [Course-provided]
# Authenticate with HuggingFace Hub to access the gated Llama-3.2-1B-Instruct model.
from huggingface_hub import login
login()

### Load a 4-bit quantized `Llama-3.2-1B-Instruct` model and its tokenizer

In [ ]:
# [Course-provided]
# Load the 4-bit NF4-quantized Llama-3.2-1B-Instruct model and its tokenizer onto the GPU.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded on device: {device}")
print(f"Model dtype: {model.dtype}")

### Test your quantized model with different prompts

In [ ]:
# [My implementation]
# Test 3 prompts (at least one UW-Madison related).
def generate_response(prompt, model, tokenizer, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# FILL THE 3 PROMPTS HERE
prompts = [
    ## 3 PROMPTS
]

assert len(prompts) == 3

for p in prompts:
    print(f"Prompt: {p}")
    response = generate_response(p, model, tokenizer)
    print(f"Response: {response}")
    print("-" * 80)

### Analyze the prompt for which the model fails.

In [ ]:
# [Course-provided]
# Run the failing prompt
failing_prompt = "What were the exact exam questions on the CS 639 midterm at UW-Madison in Spring 2025?"
print(f"Prompt: {failing_prompt}")
response = generate_response(failing_prompt, model, tokenizer)
print(f"Response: {response}")

### Enhance model responses using chat templates

In [ ]:
# [Course-provided]
# Apply a role-playing system prompt via chat template and generate a response.
messages = [
    {
        "role": "system",
        "content": "You are a medieval scholar who speaks in archaic English and uses phrases like 'forsooth' and 'verily' frequently."
    },
    {
        "role": "user",
        "content": "What is the internet?"
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"Response: {response}")

### Test the model before fine-tuning

In [ ]:
# [Course-provided]
# Baseline inference on a CS 639 course-specific prompt before any fine-tuning.
course_prompt = "What NoSQL databases are covered in the CS 639 course?"

messages_course = [
    {
        "role": "system",
        "content": "You are an instructor of CS 639 Data Management for Data Science course at UW-Madison, and are currently answering student questions."
    },
    {
        "role": "user",
        "content": course_prompt
    }
]

formatted_course_prompt = tokenizer.apply_chat_template(
    messages_course,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_course_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response_before = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\nResponse (Before Fine-Tuning):\n{response_before}")


### Pre-process the lecture transcripts

In [ ]:
# [My implementation]
# Define clean_transcript(), fixed_size_chunks(), and formatting helpers; build raw transcript chunks.
import os, re, random
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Step 1: clean ────────────────────────────────────────────────────────────
def clean_transcript(text: str) -> str:
    r"""
    Remove common noise artifacts from auto-generated lecture transcripts.

    The raw .txt files contain two types of noise:

    1. Background noise markers — phrases like '[BACKGROUND]' or '[background]'
       (match case-insensitively). Remove them entirely with re.sub.
         Pattern : r'\[BACKGROUND\]'
         Flags   : re.IGNORECASE

    2. Speaker-turn prefixes — lines begin with '>>' to mark a new speaker, e.g.
         '>> So today we cover SQL.'
       Strip the leading '>>' and any surrounding whitespace from every line.
         Per-line pattern: r'^\s*>>\s*'

    After the above, normalise whitespace:
    3. Strip leading/trailing whitespace from each individual line.
    4. Collapse runs of 3+ consecutive newlines into exactly 2 (one blank line).
         Pattern: r'\n{3,}'  →  '\n\n'
    5. Strip the final string.

    Args:
        text: Raw transcript string read directly from a .txt file.

    Returns:
        Cleaned string with noise removed and whitespace normalised.

    Example:
        Input : '>> So today we cover SQL.\n[BACKGROUND]\n>> Any questions?\n\n\n\nOkay.'
        Output: 'So today we cover SQL.\n\nAny questions?\n\nOkay.'
    """
    raise NotImplementedError("Implement clean_transcript()")


# ── Step 2: chunk into fixed-size token windows ──────────────────────────────
def fixed_size_chunks(text: str, chunk_size: int = 512, overlap: int = 50):
    """
    Split text into overlapping fixed-size chunks measured in whitespace tokens.

    Fine-tuning truncates any training example longer than max_seq_length.
    Chunking first ensures that every part of every transcript is trained on.

    Algorithm — sliding window:
        1. Tokenise by splitting on whitespace:
               tokens = text.split()
        2. Compute the step (how far the window advances each iteration):
               step = chunk_size - overlap
           The overlap keeps 'overlap' tokens shared between consecutive chunks
           so that concepts at boundaries are not split across two examples.
        3. Iterate i = 0, step, 2*step, … while i < len(tokens):
               window = tokens[i : i + chunk_size]
               if window is non-empty, append ' '.join(window) to results.
        4. Return the list of chunk strings.

    Args:
        text:       Cleaned transcript string (output of clean_transcript).
        chunk_size: Max tokens per chunk (default 512 — fits model max_seq_length).
        overlap:    Tokens shared between adjacent chunks (default 50).
                    Must be strictly less than chunk_size.

    Returns:
        List[str] — each element is a whitespace-joined token window.
        Returns [] if text is empty.

    Example  (chunk_size=4, overlap=1, step=3):
        tokens : ['a', 'b', 'c', 'd', 'e', 'f', 'g']
        Chunks : ['a b c d', 'd e f g', 'g']
    """
    raise NotImplementedError("Implement fixed_size_chunks()")


# ── Step 3: collect raw chunks with topic metadata ───────────────────────────
transcript_dir = 'transcripts/transcripts'
raw_chunks = []  # list of {"chunk": str, "topic": str}

for fname in sorted(os.listdir(transcript_dir)):
    if not fname.endswith('.txt'):
        continue
    # extract topic from filename, e.g. "12 en-English-CS639_ SQL window functions.txt"
    topic = re.sub(r'^\d+\.?\d*\s+en-English-(?:CS639[_:]\s*|SQL\s+\d+[_:]\s*)?', '', fname)
    topic = topic.replace('.txt', '').strip()
    with open(os.path.join(transcript_dir, fname), 'r', encoding='utf-8') as f:
        raw = f.read()
    cleaned = clean_transcript(raw)
    for chunk in fixed_size_chunks(cleaned, chunk_size=512, overlap=50):
        raw_chunks.append({'chunk': chunk, 'topic': topic})

print(f'Total chunks after preprocessing: {len(raw_chunks)}')
print(f'Topics: {sorted(set(c["topic"] for c in raw_chunks))}')


### Generate QA training pairs with Qwen2.5-7B-Instruct

In [ ]:
# [Course-provided]
# Unload Section 1's Llama 1B from GPU memory to make room for Qwen2.5-7B.
import gc

# Delete the Llama 1B model and tokenizer loaded in Section 1
try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# [Course-provided]
# Load 4-bit quantized Qwen2.5-7B-Instruct teacher model (~20-30 min on T4).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_model_id = "Qwen/Qwen2.5-7B-Instruct"

qwen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_id,
    quantization_config=qwen_bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Qwen 7B loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# [Course-provided]
# Define generate_qa_pairs(): prompt Qwen2.5-7B to output 3 JSON-structured exam QA pairs per chunk (~2 hrs).
import json

QA_SYSTEM_PROMPT = """You are an expert educator creating exam-style questions from lecture transcripts for a CS 639 (Data Management for Data Science) course at UW-Madison."""

QA_USER_TEMPLATE = """Given the following lecture excerpt on "{topic}", generate exactly 3 question-answer pairs suitable for a university exam.

Requirements:
- Questions should be specific and test understanding (not just recall)
- Answers must be self-contained — a student should understand the answer without reading the transcript
- Answers should be 2-4 sentences each
- Cover different aspects of the content in the excerpt
- Output ONLY a valid JSON array, no other text: [{{"question": "...", "answer": "..."}}, ...]

Lecture excerpt:
{chunk}"""


def generate_qa_pairs(chunk: str, topic: str, max_new_tokens: int = 1024) -> list:
    """Generate QA pairs from a transcript chunk using Qwen 7B."""
    messages = [
        {"role": "system", "content": QA_SYSTEM_PROMPT},
        {"role": "user", "content": QA_USER_TEMPLATE.format(topic=topic, chunk=chunk)},
    ]
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    generated = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

    # Parse JSON — try direct parse first, then extract JSON array with regex
    try:
        pairs = json.loads(generated)
        if isinstance(pairs, list):
            return [p for p in pairs if "question" in p and "answer" in p]
    except json.JSONDecodeError:
        pass

    # Fallback: extract JSON array from response
    match = re.search(r'\[.*\]', generated, re.DOTALL)
    if match:
        try:
            pairs = json.loads(match.group())
            if isinstance(pairs, list):
                return [p for p in pairs if "question" in p and "answer" in p]
        except json.JSONDecodeError:
            pass

    return []


# Quick test on one chunk
test_pairs = generate_qa_pairs(raw_chunks[0]["chunk"], raw_chunks[0]["topic"])
print(f"Test: generated {len(test_pairs)} QA pairs from chunk about '{raw_chunks[0]['topic']}'")
for i, p in enumerate(test_pairs):
    print(f"\n  Q{i+1}: {p['question']}")
    print(f"  A{i+1}: {p['answer']}...")

In [ ]:
# [Course-provided]
# Run generate_qa_pairs() over all transcript chunks; save results to qa_pairs.json.
from tqdm import tqdm

all_qa_pairs = []
failed_chunks = 0

for item in tqdm(raw_chunks, desc="Generating QA pairs"):
    pairs = generate_qa_pairs(item["chunk"], item["topic"])
    if pairs:
        for p in pairs:
            p["topic"] = item["topic"]
        all_qa_pairs.extend(pairs)
    else:
        failed_chunks += 1

print(f"\nGeneration complete!")
print(f"  Total QA pairs: {len(all_qa_pairs)}")
print(f"  Failed chunks (no valid JSON): {failed_chunks}/{len(raw_chunks)}")

# Topic distribution
from collections import Counter
topic_counts = Counter(p["topic"] for p in all_qa_pairs)
print(f"\nQA pairs per topic:")
for topic, count in topic_counts.most_common():
    print(f"  {topic}: {count}")

# Save to disk for reproducibility
with open("qa_pairs.json", "w") as f:
    json.dump(all_qa_pairs, f, indent=2)
print(f"\nSaved {len(all_qa_pairs)} QA pairs to qa_pairs.json")

In [ ]:
# [Course-provided]
# Preview a random sample of generated QA pairs to verify output quality.
print("=" * 70)
print("Sample QA pairs:")
print("=" * 70)
for i, pair in enumerate(random.sample(all_qa_pairs, min(5, len(all_qa_pairs)))):
    print(f"\n[{pair['topic']}]")
    print(f"  Q: {pair['question']}")
    print(f"  A: {pair['answer']}")
    print("-" * 70)

### LoRA fine-tuning model using generated QA pairs

In [ ]:
# [Course-provided]
# Unload Qwen2.5-7B, reload Llama-3.2-1B-Instruct, and format QA pairs as chat-template training examples.
model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Llama 1B reloaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Format QA pairs as chat-template training examples ────────────────────────
SYSTEM_MSG = (
    'You are an instructor of CS 639 Data Management for Data Science '
    'at UW-Madison. Answer student questions using the course material below.'
)

# (OPTIONAL) reload from disk if restarting from this cell
with open("qa_pairs.json") as f:
    all_qa_pairs = json.load(f)

all_texts = []
for pair in all_qa_pairs:
    formatted = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": pair["question"]},
            {"role": "assistant", "content": pair["answer"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    all_texts.append({"text": formatted})

print(f"\nFormatted {len(all_texts)} QA training examples (from {len(all_qa_pairs)} QA pairs)")

In [ ]:
# [Course-provided]
# Shuffle all_texts and split 90/10 into HuggingFace Dataset objects for training and evaluation.
from datasets import Dataset
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

random.seed(42)
random.shuffle(all_texts)

split        = int(len(all_texts) * 0.9)
train_data   = all_texts[:split]
test_data    = all_texts[split:]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')

In [ ]:
# [Course-provided]
# Configure LoRA (r=8), TrainingArguments, and SFTTrainer; fine-tune Llama 1B on the generated QA dataset. (~30 mins on T4)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


tokenizer.model_max_length = 512

lora_config = LoraConfig(
    r=8,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "o_proj", "k_proj", "v_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

training_args = TrainingArguments(
    eval_strategy="steps",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    logging_first_step=True,
    output_dir="./results",
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=lora_config,
)

trainer.train()

### Test the model after fine-tuning

In [ ]:
# [Course-provided]
# Run the same course prompt on the base model (adapter off) vs fine-tuned model (adapter on).
trainer.model.eval()

course_prompt = "What No SQL database was covered in the course?"
_messages = [
    {"role": "system",
     "content": "You are an instructor of CS 639 Data Management for Data Science "
                "course at UW-Madison, and are currently answering student questions."},
    {"role": "user",
     "content": course_prompt},
]
_prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)

gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,          # greedy — deterministic, easier to compare
    repetition_penalty=1.3,   # penalise repeated tokens to avoid looping
    no_repeat_ngram_size=3,   # block any 3-gram from appearing twice
)

# ── Base model (adapter disabled) ────────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with trainer.model.disable_adapter():
    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)
response_base = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# ── Fine-tuned model (adapter active) ────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = trainer.model.generate(**inputs, **gen_kwargs)
response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\n{'─'*60}")
print(f"Base model (no fine-tuning):\n{response_base}")
print(f"\n{'─'*60}")
print(f"Fine-tuned model:\n{response_ft}")


In [ ]:
# [My implementation]
# Q2.5 Come up with 3-5 prompt questions that you and run your finetined model on it. 
# Later, we will use the same prompts to compare against RAG pipeline.

trainer.model.eval()

# List of prompts to iterate over
course_prompts = [
    # List your prompts here
]

# Generation kwargs defined once outside the loop
gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,          
    repetition_penalty=1.3,   
    no_repeat_ngram_size=3,   
)

for course_prompt in course_prompts:
    _messages = [
        {"role": "system",
         "content": "You are an instructor of CS 639 Data Management for Data Science "
                    "course at UW-Madison, and are currently answering student questions."},
        {"role": "user",
         "content": course_prompt},
    ]
    
    # Format and tokenize the prompt
    _prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(_prompt, return_tensors="pt").to(device)
    
    # ── Fine-tuned model (adapter active) ────────────────────────────────────
    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)
        
    # Decode only the newly generated tokens
    response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # Print the results
    print(f"Prompt: {course_prompt}")
    print(f"{'─'*60}")
    print(f"Fine-tuned model:\n{response_ft}")
    print(f"{'='*60}\n")

### Evaluate fine-tuning quantitatively using perplexity

In [ ]:
# [Course-provided]
# Compute perplexity (exp of avg cross-entropy loss) on the test set for base and fine-tuned models.
import math
from torch.nn import CrossEntropyLoss

def compute_perplexity(model, tokenizer, texts, device, max_length=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt",
                            truncation=True, max_length=max_length).to(device)
            labels = enc["input_ids"].clone()
            out = model(**enc, labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_loss += out.loss.item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_loss / total_tokens)

test_texts = [item["text"] for item in test_data]

print("Computing perplexity on base model...")
with trainer.model.disable_adapter():
    base_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Base model perplexity: {base_ppl:.2f}")

print("\nComputing perplexity on fine-tuned model...")
ft_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Fine-tuned model perplexity: {ft_ppl:.2f}")

improvement = ((base_ppl - ft_ppl) / base_ppl) * 100
print(f"\nPercentage improvement: {improvement:.2f}%")

### Setup: Credentials and helper functions

In [ ]:
# [Course-provided]
# Reset variables, enter credentials, define transcript helpers, and load all 23 transcripts.
# %reset -f clears all Python variables to free CPU memory before loading Section 3 libraries.
# Pip-installed packages are NOT removed — you do NOT need to reinstall them.
# If you get an ImportError after %reset -f, restart the runtime and re-run the install cell.
%reset -f
import os, re, getpass
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

HF_TOKEN    = getpass.getpass("HuggingFace API token: ")
os.environ["HF_API_TOKEN"] = HF_TOKEN

# Elastic Cloud credentials
ES_CLOUD_ID = getpass.getpass("Elastic Cloud ID: ")
ES_API_KEY  = getpass.getpass("Elastic Cloud API key: ")

In [ ]:
# [My implementation]
# Implement sentence_chunks(); then load all 23 raw transcripts.

def sentence_chunks(text: str, n: int = 5):
    """
    Split cleaned transcript text into non-overlapping groups of consecutive sentences.

    Uses NLTK's sentence tokeniser (``nltk.sent_tokenize``) to detect sentence
    boundaries, then groups the resulting sentences into windows of at most ``n``
    sentences each. This preserves natural language units, which improves BM25
    retrieval precision compared with fixed-size word windows: keywords that span
    a sentence boundary are never split across two chunks.

    Algorithm
    ---------
    1. Tokenise the text into sentences:
           sentences = nltk.sent_tokenize(text)
    2. Iterate with step ``n`` over the sentence list:
           for i in range(0, len(sentences), n):
               window = sentences[i : i + n]
    3. For each non-empty window, join with a single space and strip whitespace:
           chunk = ' '.join(window).strip()
    4. Collect and return all non-empty chunks as a list of strings.

    Args:
        text: Cleaned transcript string (output of ``clean_transcript``).
        n:    Maximum number of sentences per chunk. Default 5.
              Must be a positive integer.

    Returns:
        List[str]: Each element is a space-joined group of up to ``n`` consecutive
        sentences. Returns [] if ``text`` contains no detectable sentences.

    Example (n=2):
        Input  : "Dogs bark. Cats meow. Birds sing."
        Chunks : ["Dogs bark. Cats meow.", "Birds sing."]
    """
    raise NotImplementedError("Implement sentence_chunks()")


transcript_dir = "transcripts/transcripts"
raw_transcripts = []
for fname in sorted(os.listdir(transcript_dir)):
    if fname.endswith(".txt"):
        with open(os.path.join(transcript_dir, fname), "r", encoding="utf-8") as f:
            raw_transcripts.append((fname, f.read()))
print(f"Loaded {len(raw_transcripts)} transcripts.")


In [ ]:
# [Course-provided]
# Factory that returns an ElasticsearchDocumentStore connected to Elastic Cloud.
from haystack import Document
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore

def make_document_store(index_name):
    return ElasticsearchDocumentStore(
        hosts=None,
        cloud_id=ES_CLOUD_ID,
        api_key=ES_API_KEY,
        index=index_name
    )

### Part A — Fixed-size chunking → `transcripts_fixed`

In [ ]:
# [My implementation]
# Implement load_fixed_chunks(): clean -> 512-token overlapping windows -> index into ES.
def load_fixed_chunks(ds, raw_transcripts, chunk_size=512, overlap=50):
    """
    Load fixed-size transcript chunks into an Elasticsearch document store.

    Processes every raw transcript through a four-step pipeline and writes the
    resulting Haystack Document objects into the provided document store. If the
    store already contains documents from a previous run, they are deleted first
    so the index is always rebuilt from scratch.

    Pipeline
    --------
    1. Clean
       Call ``clean_transcript(raw)`` on each raw transcript string to strip
       noise markers such as ``[BACKGROUND]`` and speaker-turn prefixes ``>>``.

    2. Chunk
       Call ``fixed_size_chunks(cleaned, chunk_size=chunk_size, overlap=overlap)``
       to split the cleaned text into overlapping windows of whitespace tokens.
       Each window is ``chunk_size`` tokens long; consecutive windows share
       ``overlap`` tokens so that concepts near a boundary appear in both
       neighbouring chunks.

    3. Wrap
       For every chunk string, construct a ``Document`` (imported from haystack)
       with two fields:
           content : the chunk string
           meta    : a dict containing
                         "source"   – transcript filename without the ".txt" suffix
                         "chunk_id" – 0-based integer index of the chunk within
                                      that transcript

    4. Delete existing documents
       Before writing, check whether the index is non-empty:
           if ds.count_documents() > 0:
               existing = ds.filter_documents()
               ds.delete_documents(document_ids=[doc.id for doc in existing])
       This guarantees a clean reload on every call — no stale chunks remain.

    5. Write
       Call ``ds.write_documents(docs)`` to bulk-index all Document objects.

    Parameters
    ----------
    ds : ElasticsearchDocumentStore
        Target Elasticsearch index to write into.
    raw_transcripts : list[tuple[str, str]]
        List of (filename, raw_text) pairs, e.g. as produced by reading each
        ".txt" file from the transcripts directory.
    chunk_size : int, optional
        Number of whitespace-split tokens per chunk. Default 512.
    overlap : int, optional
        Tokens shared between adjacent chunks. Must be < chunk_size. Default 50.

    Returns
    -------
    int
        Total number of documents now present in the store after writing.

    Example
    -------
    ds_fixed = make_document_store("transcripts_fixed")
    n = load_fixed_chunks(ds_fixed, raw_transcripts)
    print(f"Indexed {n} fixed-size chunks")
    """
    raise NotImplementedError("Implement load_fixed_chunks()")


ds_fixed = make_document_store("transcripts_fixed")
n_fixed  = load_fixed_chunks(ds_fixed, raw_transcripts)
print(f"Fixed-size index: {n_fixed} chunks")


### Part B — Sentence-level chunking → `transcripts_sentence`

In [ ]:
# [My implementation]
# Implement load_sentence_chunks(): clean -> 5-sentence groups -> index into ES.
def load_sentence_chunks(ds, raw_transcripts, n=5):
    """
    Load sentence-grouped transcript chunks into an Elasticsearch document store.

    Processes every raw transcript through a four-step pipeline and writes the
    resulting Haystack Document objects into the provided document store. If the
    store already contains documents from a previous run, they are deleted first
    so the index is always rebuilt from scratch.

    Unlike the fixed-size strategy (Part A), this approach preserves natural
    sentence boundaries. Each chunk is a grammatically complete unit, which tends
    to improve BM25 precision: keywords that appear mid-sentence are never split
    across two chunks, so retrieval is more likely to return the full context
    surrounding a relevant term.

    Pipeline
    --------
    1. Clean
       Call ``clean_transcript(raw)`` on each raw transcript string to strip
       noise markers such as ``[BACKGROUND]`` and speaker-turn prefixes ``>>``.

    2. Chunk
       Call ``sentence_chunks(cleaned, n=n)`` to split the cleaned text into
       groups of at most ``n`` consecutive sentences. Sentence boundaries are
       detected by ``nltk.sent_tokenize``. Each group is joined with a single
       space. There is no overlap between groups.

    3. Wrap
       For every chunk string, construct a ``Document`` (imported from haystack)
       with two fields:
           content : the chunk string (up to n sentences)
           meta    : a dict containing
                         "source"   – transcript filename without the ".txt" suffix
                         "chunk_id" – 0-based integer index of the chunk within
                                      that transcript

    4. Delete existing documents
       Before writing, check whether the index is non-empty:
       This guarantees a clean reload on every call — no stale chunks remain.

    5. Write
       Call ``ds.write_documents(docs)`` to bulk-index all Document objects.

    Parameters
    ----------
    ds : ElasticsearchDocumentStore
        Target Elasticsearch index to write into.
    raw_transcripts : list[tuple[str, str]]
        List of (filename, raw_text) pairs, e.g. as produced by reading each
        ".txt" file from the transcripts directory.
    n : int, optional
        Maximum number of sentences per chunk. Default 5.

    Returns
    -------
    int
        Total number of documents now present in the store after writing.

    Example
    -------
    ds_sentence = make_document_store("transcripts_sentence")
    n = load_sentence_chunks(ds_sentence, raw_transcripts)
    print(f"Indexed {n} sentence-level chunks")
    """
    raise NotImplementedError("Implement load_sentence_chunks()")

ds_sentence = make_document_store("transcripts_sentence")
n_sentence  = load_sentence_chunks(ds_sentence, raw_transcripts)
print(f"Sentence-level index: {n_sentence} chunks")


### Part C — Retrieval precision comparison (Precision@3)

In [ ]:
# [My implementation]
# Retrieve top-3 docs per exam question from both indexes; compute Precision@3 interactively.
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever

retriever_fixed    = ElasticsearchBM25Retriever(document_store=ds_fixed,    top_k=3)
retriever_sentence = ElasticsearchBM25Retriever(document_store=ds_sentence, top_k=3)

exam_questions = [
    "What are window functions in SQL?",
    "What is relational algebra?",
    "What is a SQL subquery and when would you use one?"
    # Add 2 more of your own
]

assert len(exam_questions) == 5

def precision_at_k(retrieved_docs, query, k=3):
    relevant = 0
    for i, doc in enumerate(retrieved_docs[:k]):
        print(f"\n  Chunk {i+1}: {doc.content[:200]}...")
        label = input(f"  Relevant to '{query}'? (1=yes, 0=no): ").strip()
        relevant += int(label) if label in ("0", "1") else 0
    return relevant / k

results_fixed    = {}
results_sentence = {}

for q in exam_questions:
    print(f"\n{'='*60}\nQuestion: {q}\n{'='*60}")
    docs_fixed = retriever_fixed.run(query=q)["documents"]
    print("\n-- Fixed-size chunks --")
    results_fixed[q] = precision_at_k(docs_fixed, q)
    docs_sentence = retriever_sentence.run(query=q)["documents"]
    print("\n-- Sentence-level chunks --")
    results_sentence[q] = precision_at_k(docs_sentence, q)

print("\n\nPrecision@3 Results:")
print(f"{'Question':<45} {'Fixed P@3':>10} {'Sentence P@3':>13}")
print("-" * 70)
for q in exam_questions:
    print(f"{q:<45} {results_fixed[q]:>10.2f} {results_sentence[q]:>13.2f}")
avg_fixed    = sum(results_fixed.values()) / len(exam_questions)
avg_sentence = sum(results_sentence.values()) / len(exam_questions)
print(f"{'Average':<45} {avg_fixed:>10.2f} {avg_sentence:>13.2f}")

In [ ]:
# [Course-provided]
# Build Haystack RAG pipeline: BM25 retriever -> ChatPromptBuilder -> Qwen via HF Inference API.
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

document_store = make_document_store("transcripts_sentence")

template = """
{% message role=\"system\" %}
You are a helpful assistant.
{% endmessage %}

{% message role=\"user\" %}
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}

Question: {{ query }}?
{% endmessage %}
"""

rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
rag_pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"},
))
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

# Quick test
test_q = "Give me everything that was covered for SQL"
result = rag_pipeline.run({"prompt_builder": {"query": test_q}, "retriever": {"query": test_q}})
try:
    print(result["llm"]["replies"][0].content)
except AttributeError:
    print(result["llm"]["replies"][0])

In [ ]:
# [Course-provided]
# Write app.py: Streamlit chatbot with BM25 retrieval and expandable retrieved-docs sidebar.
%%writefile app.py
import os
import streamlit as st
from haystack import Pipeline
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

ES_CLOUD_ID = os.environ.get("ES_CLOUD_ID")
ES_API_KEY  = os.environ.get("ES_API_KEY")
INDEX_NAME  = "transcripts_sentence"

@st.cache_resource
def build_rag_pipeline():
    document_store = ElasticsearchDocumentStore(
        hosts=None, cloud_id=ES_CLOUD_ID, api_key=ES_API_KEY, index=INDEX_NAME
    )
    template = """
{% message role="system" %}
You are a helpful assistant.
{% endmessage %}
{% message role="user" %}
Given the following information, answer the question.
Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}
Question: {{ query }}?
{% endmessage %}
"""
    pipeline = Pipeline()
    pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
    pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
    pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
        api_type="serverless_inference_api", api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"} #"Qwen/Qwen2.5-7B-Instruct"
    ))
    pipeline.connect("retriever.documents", "prompt_builder.documents")
    pipeline.connect("prompt_builder.prompt", "llm.messages")
    return pipeline

rag_pipeline = build_rag_pipeline()

st.title("Course Chatbot")
st.caption("Interactive Q&A with Elasticsearch, Haystack, and HuggingFace")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask a question about the course transcripts"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    result = rag_pipeline.run({"prompt_builder": {"query": prompt}, "retriever": {"query": prompt}})
    try:
        response_text = result["llm"]["replies"][0].content
    except AttributeError:
        response_text = result["llm"]["replies"][0].text
    retrieved_docs = result.get("retriever", {}).get("documents", [])
    st.session_state.messages.append({"role": "assistant", "content": response_text})
    st.chat_message("assistant").write(response_text)
    if retrieved_docs:
        with st.expander("📄 Top 3 Retrieved Documents (BM25 Scores)"):
            for i, doc in enumerate(retrieved_docs):
                st.markdown(f"**Document {i+1}** — Source: `{doc.meta.get('source','N/A')}` | Chunk: `{doc.meta.get('chunk_id','N/A')}` | **BM25 Score: {doc.score:.3f}**")
                st.write(doc.content)
                st.divider()

In [ ]:
# [Course-provided]
# Export ES credentials as env vars, get LocalTunnel password, and launch Streamlit on port 8501.
import os
os.environ["ES_CLOUD_ID"] = ES_CLOUD_ID
os.environ["ES_API_KEY"]  = ES_API_KEY

# Get LocalTunnel password
!curl https://loca.lt/mytunnelpassword
print("\n")

# Launch Streamlit
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

In [ ]:
# [My implementation]
# Run some(3-5) test prompts through the RAG pipeline to compare against the fine-tuned model output. 
test_prompts = [
    # Use the same prompts as used in Q2.5
]

for prompt in test_prompts:
    print(f"\n{'='*70}\nPrompt: {prompt}\n{'='*70}")
    result = rag_pipeline.run({
        "prompt_builder": {"query": prompt},
        "retriever": {"query": prompt}
    })
    try:
        print(f"RAG Response:\n{result['llm']['replies'][0].content}")
    except AttributeError:
        print(f"RAG Response:\n{result['llm']['replies'][0]}")